In [1]:
import json

simple_wiki_data_path = "simple_wiki_raw_data.json"

with open(simple_wiki_data_path,"r") as file:
    simple_wiki_data = json.load(file)

simple_wiki_data

[{'url': 'https://simple.wikipedia.org/wiki/Derivative_(mathematics)',
  'title': 'Derivative (mathematics)',
  'sections': [{'heading': 'Introduction',
    'paragraphs': ['In mathematics (particularly in differential calculus ), the derivative is a way to show how steep a function is at a given point. Derivatives are similar to the slope of a line, but can be used for other curves as well. They are sometimes called the " instantaneous rate of change " of a function.',
     'More specifically, the derivative is how much a function is changing at one given point. For functions that act on the real numbers, it is the slope of the tangent line at a point on a graph. The derivative is often written as ${\\displaystyle {\\tfrac {dy}{dx}}}$ ("dy over dx" or "dy upon dx", meaning the difference in y divided by the difference in x). The d is not a variable, and therefore cannot be cancelled out. Another common notation is ${\\displaystyle f\'(x)}$ —the derivative of function ${\\displaystyle f

In [2]:
page_titles = [page.get('title') for page in simple_wiki_data]
len(page_titles), page_titles

(3, ['Derivative (mathematics)', 'Nim', "Newton's method"])

In [3]:
# no_title_items = [item for item in simple_wiki_data if item.get("title") is None]
# no_title_items

# # don't understand why these have no section
# no_sections_items = [item for item in simple_wiki_data if not item.get("sections")]
# no_sections_items

In [4]:
base_url = "https://en.wikipedia.org/wiki/"

def simple2normalwiki_url(simple_url):
    page_url_segment = simple_url.split("/")[-1]
    normalwiki_base = "https://en.wikipedia.org/wiki/"
    normalwiki_url = normalwiki_base + page_url_segment
    return normalwiki_url

test_simple_url = "https://simple.wikipedia.org/wiki/Numerical_integration"
test_normal_url = simple2normalwiki_url(test_simple_url)
test_normal_url

'https://en.wikipedia.org/wiki/Numerical_integration'

In [5]:
from bs4 import BeautifulSoup, Tag
import requests
import re
from IPython.display import Markdown, display

# --- Config ---
IGNORE_CLASSES = ["sidebar-list", "navbar", "infobox", "toc"]
STOP_SECTIONS = {"references", "external links", "see also", "notes"}

# --- Paragraph cleaning ---
def clean_paragraph(el: Tag):
    """
    Clean paragraph text from Wikipedia, preserving formulas as LaTeX.
    """
    # Remove citation superscripts
    for sup in el.find_all("sup"):
        sup.decompose()

    # Replace <math> elements with LaTeX
    for math in el.find_all("math"):
        latex = math.get("alttext") or math.get_text()
        latex = latex.strip()
        # Determine if the math is the only content of the paragraph
        if len(el.get_text(strip=True)) == len(math.get_text(strip=True)):
            math.replace_with(f"$$\n{latex}\n$$") # display math
        else:
            math.replace_with(f"${latex}$") # inline math

    # Handle lists
    if el.name in ["ul", "ol"]:
        items = []
        for i, li in enumerate(el.find_all("li", recursive=False), start=1):
            li_text = clean_paragraph(li)
            if el.name == "ul":
                items.append(f"- {li_text}")
            else:
                items.append(f"{i}) {li_text}")
        return "\n".join(items)

    # Other text
    text = el.get_text(" ", strip=True)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\s+([.,;:!?])', r'\1', text)
    return text.strip()

# Ignore section identifier
def is_in_ignored(el: Tag):
    """Return True if element is inside sidebar, infobox, navbar, or TOC."""
    for parent in el.parents:
        if any(cls in parent.get("class", []) for cls in IGNORE_CLASSES):
            return True
    return False

# --- Normal Wiki page scraper (main) ---
def scrape_normal_wiki(url: str):
    headers = {"User-Agent": "ReverseMentorBot/0.1"}
    res = requests.get(url, headers=headers)
    soup = BeautifulSoup(res.text, "html.parser")
    content = soup.find("div", class_="mw-parser-output")
    if not content:
        return {"error": "content not found"}

    sections = []

    # --- Intro paragraphs (before first H2) ---
    intro_paragraphs = []
    for el in content.find_all(["p", "li", "dd", "ul", "ol", "h2", "h3", "h4", "h5"]):
        if el.name.startswith("h2"):
            break
        if el.name not in ["p", "li", "dd", "ul", "ol"]:
            continue
        if is_in_ignored(el):
            continue
        text = clean_paragraph(el)
        if text:
            intro_paragraphs.append(text)
    if intro_paragraphs:
        sections.append({"heading": "Introduction", "paragraphs": intro_paragraphs})

    # --- Process headings (flat, no nested subsections) ---
    current_section = None
    for el in content.find_all(["p", "li", "dd", "ul", "ol", "h2", "h3", "h4", "h5"]):
        if el.name.startswith("h"):
            heading = el.get_text(" ", strip=True).replace("[edit]", "")
            if heading.lower() in STOP_SECTIONS:
                break
            current_section = {"heading": heading, "paragraphs": []}
            sections.append(current_section)
            continue

        # Content elements
        if el.name in ["p", "li", "dd", "ul", "ol"]:
            if is_in_ignored(el) or current_section is None:
                continue
            text = clean_paragraph(el)
            if text:
                current_section["paragraphs"].append(text)

    title = soup.find("h1").get_text(strip=True) if soup.find("h1") else None
    return {"url": url, 
            "title": title, 
            "sections": sections
            }



In [6]:
# --- Test scraper ---
test_normal_url = 'https://en.wikipedia.org/wiki/Numerical_integration'
test = scrape_normal_wiki(test_normal_url)
test

{'url': 'https://en.wikipedia.org/wiki/Numerical_integration',
 'title': 'Numerical integration',
 'sections': [{'heading': 'Introduction',
   'paragraphs': ['In analysis, numerical integration comprises a broad family of algorithms for calculating the numerical value of a definite integral. The term numerical quadrature (often abbreviated to quadrature ) is more or less a synonym for "numerical integration", especially as applied to one-dimensional integrals. Some authors refer to numerical integration over more than one dimension as cubature; others take "quadrature" to include higher-dimensional integration.',
    'The basic problem in numerical integration is to compute an approximate solution to a definite integral',
    '$$ {\\displaystyle \\int _{a}^{b}f(x)\\,dx} $$',
    'to a given degree of accuracy. If f ( x ) is a smooth function integrated over a small number of dimensions, and the domain of integration is bounded, there are many methods for approximating the integral to t

In [17]:
# --- Pretty print ---
def pretty_print_sections(data):
    md = f"# {data['title']}\n\n"
    
    for section in data['sections']:
        md += f"## {section['heading']}\n\n"
        for para in section["paragraphs"]:
            md += para + "\n\n"
    md += "\n------------------------------------------------------------------"
    display(Markdown(md))
    
pretty_print_sections(test)

# Numerical integration

## Introduction

In analysis, numerical integration comprises a broad family of algorithms for calculating the numerical value of a definite integral. The term numerical quadrature (often abbreviated to quadrature ) is more or less a synonym for "numerical integration", especially as applied to one-dimensional integrals. Some authors refer to numerical integration over more than one dimension as cubature; others take "quadrature" to include higher-dimensional integration.

The basic problem in numerical integration is to compute an approximate solution to a definite integral

$$ {\displaystyle \int _{a}^{b}f(x)\,dx} $$

to a given degree of accuracy. If f ( x ) is a smooth function integrated over a small number of dimensions, and the domain of integration is bounded, there are many methods for approximating the integral to the desired precision.

Numerical integration has roots in the geometrical problem of finding a square with the same area as a given plane figure ( quadrature or squaring ), as in the quadrature of the circle. The term is also sometimes used to describe the numerical solution of differential equations.

## Motivation and need

There are several reasons for carrying out numerical integration, as opposed to analytical integration by finding the antiderivative:

1) The integrand f ( x ) may be known only at certain points, such as obtained by sampling. Some embedded systems and other computer applications may need numerical integration for this reason.
2) A formula for the integrand may be known, but it may be difficult or impossible to find an antiderivative that is an elementary function. An example of such an integrand is f ( x ) = exp (− x ), the antiderivative of which (the error function, times a constant) cannot be written in elementary form (see also: Nonelementary integral ).
3) It may be possible to find an antiderivative symbolically, but it may be easier to compute a numerical approximation than to compute the antiderivative. That may be the case if the antiderivative is given as an infinite series or product, or if its evaluation requires a special function that is not available.

The integrand f ( x ) may be known only at certain points, such as obtained by sampling. Some embedded systems and other computer applications may need numerical integration for this reason.

A formula for the integrand may be known, but it may be difficult or impossible to find an antiderivative that is an elementary function. An example of such an integrand is f ( x ) = exp (− x ), the antiderivative of which (the error function, times a constant) cannot be written in elementary form (see also: Nonelementary integral ).

It may be possible to find an antiderivative symbolically, but it may be easier to compute a numerical approximation than to compute the antiderivative. That may be the case if the antiderivative is given as an infinite series or product, or if its evaluation requires a special function that is not available.

## History

The term "numerical integration" first appears in 1915 in the publication A Course in Interpolation and Numeric Integration for the Mathematical Laboratory by David Gibb.

"Quadrature" is a historical mathematical term that means calculating area. Quadrature problems have served as one of the main sources of mathematical analysis. Mathematicians of Ancient Greece, according to the Pythagorean doctrine, understood calculation of area as the process of constructing geometrically a square having the same area ( squaring ) — that is why the process was named "quadrature". Examples include quadrature of the circle, Lune of Hippocrates, and the treatise Quadrature of the Parabola. This construction must be performed only by means of compass and straightedge.

The ancient Babylonians used the trapezoidal rule to integrate the motion of Jupiter along the ecliptic.

For a quadrature of a rectangle with the sides a and b it is necessary to construct a square with the side ${\displaystyle x={\sqrt {ab}}}$ (the geometric mean of a and b ). For this purpose it is possible to use the following fact: if we draw the circle with the sum of a and b as the diameter, then the height BH (from a point of their connection to crossing with a circle) equals their geometric mean. The similar geometrical construction solves a problem of a quadrature for a parallelogram and a triangle.

Problems of quadrature for curvilinear figures are much more difficult. The quadrature of the circle with compass and straightedge had been proved in the 19th century to be impossible. Nevertheless, for some figures (for example the lune of Hippocrates ) a quadrature can be performed. The quadratures of a sphere surface and a parabola segment done by Archimedes became the highest achievement of the antique analysis.

- The area of the surface of a sphere is equal to quadruple the area of a great circle of this sphere.
- The area of a segment of the parabola cut from it by a straight line is 4/3 the area of the triangle inscribed in this segment.

The area of the surface of a sphere is equal to quadruple the area of a great circle of this sphere.

The area of a segment of the parabola cut from it by a straight line is 4/3 the area of the triangle inscribed in this segment.

To prove the results, Archimedes used the method of exhaustion of Eudoxus.

In medieval Europe the quadrature meant calculation of area by any method. More often the method of indivisibles was used; it was less rigorous, but more simple and powerful. With its help Galileo Galilei and Gilles de Roberval found the area of a cycloid arch, Grégoire de Saint-Vincent investigated the area under a hyperbola ( Opus Geometricum, 1647), and Alphonse Antonio de Sarasa, de Saint-Vincent's pupil and commentator, noted the relation of this area to logarithms.

John Wallis algebrised this method: he wrote in his Arithmetica Infinitorum (1656) series that we now call the definite integral, and he calculated their values. Isaac Barrow and James Gregory made further progress: quadratures for some algebraic curves and spirals. Christiaan Huygens successfully performed a quadrature of some solids of revolution.

The quadrature of the hyperbola by Saint-Vincent and de Sarasa provided a new function, the natural logarithm, of critical importance.

With the invention of integral calculus came a universal method for area calculation. In response, the term "quadrature" has become traditional, and instead the modern phrase " computation of a univariate definite integral " is more common.

## Methods for one-dimensional integrals

A quadrature rule is an approximation of the definite integral of a function, usually stated as a weighted sum of function values at specified points within the domain of integration.

Numerical integration methods can generally be described as combining evaluations of the integrand to get an approximation to the integral. The integrand is evaluated at a finite set of points called integration points and a weighted sum of these values is used to approximate the integral. The integration points and weights depend on the specific method used and the accuracy required from the approximation.

An important part of the analysis of any numerical integration method is to study the behavior of the approximation error as a function of the number of integrand evaluations. A method that yields a small error for a small number of evaluations is usually considered superior. Reducing the number of evaluations of the integrand reduces the number of arithmetic operations involved, and therefore reduces the total error. Also, each evaluation takes time, and the integrand may be arbitrarily complicated.

## Quadrature rules based on step functions

A "brute force" kind of numerical integration can be done, if the integrand is reasonably well-behaved (i.e. piecewise continuous and of bounded variation ), by evaluating the integrand with very small increments.

This simplest method approximates the function by a step function (a piecewise constant function, or a segmented polynomial of degree zero) that passes through the point ${\textstyle \left({\frac {a+b}{2}},f\left({\frac {a+b}{2}}\right)\right)}$. This is called the midpoint rule or rectangle rule ${\displaystyle \int _{a}^{b}f(x)\,dx\approx (b-a)f\left({\frac {a+b}{2}}\right).}$

## Quadrature rules based on interpolating functions

A large class of quadrature rules can be derived by constructing interpolating functions that are easy to integrate. Typically these interpolating functions are polynomials. In practice, since polynomials of very high degree tend to oscillate wildly, only polynomials of low degree are used, typically linear and quadratic.

The interpolating function may be a straight line (an affine function, i.e. a polynomial of degree 1) passing through the points ${\displaystyle \left(a,f(a)\right)}$ and ${\displaystyle \left(b,f(b)\right)}$. This is called the trapezoidal rule ${\displaystyle \int _{a}^{b}f(x)\,dx\approx (b-a)\left({\frac {f(a)+f(b)}{2}}\right).}$

For either one of these rules, we can make a more accurate approximation by breaking up the interval ${\displaystyle [a,b]}$ into some number ${\displaystyle n}$ of subintervals, computing an approximation for each subinterval, then adding up all the results. This is called a composite rule, extended rule, or iterated rule. For example, the composite trapezoidal rule can be stated as ${\displaystyle \int _{a}^{b}f(x)\,dx\approx {\frac {b-a}{n}}\left({f(a) \over 2}+\sum _{k=1}^{n-1}\left(f\left(a+k{\frac {b-a}{n}}\right)\right)+{f(b) \over 2}\right),}$

where the subintervals have the form ${\displaystyle [a+kh,a+(k+1)h]\subset [a,b],}$ with ${\textstyle h={\frac {b-a}{n}}}$ and ${\displaystyle k=0,\ldots,n-1.}$ Here we used subintervals of the same length ${\displaystyle h}$ but one could also use intervals of varying length ${\displaystyle \left(h_{k}\right)_{k}}$.

Interpolation with polynomials evaluated at equally spaced points in ${\displaystyle [a,b]}$ yields the Newton–Cotes formulas, of which the rectangle rule and the trapezoidal rule are examples. Simpson's rule, which is based on a polynomial of order 2, is also a Newton–Cotes formula.

Quadrature rules with equally spaced points have the very convenient property of nesting. The corresponding rule with each interval subdivided includes all the current points, so those integrand values can be re-used.

If we allow the intervals between interpolation points to vary, we find another group of quadrature formulas, such as the Gaussian quadrature formulas. A Gaussian quadrature rule is typically more accurate than a Newton–Cotes rule that uses the same number of function evaluations, if the integrand is smooth (i.e., if it is sufficiently differentiable). Other quadrature methods with varying intervals include Clenshaw–Curtis quadrature (also called Fejér quadrature) methods, which do nest.

Gaussian quadrature rules do not nest, but the related Gauss–Kronrod quadrature formulas do.

## Adaptive algorithms

Adaptive quadrature is a numerical integration method in which the integral of a function ${\displaystyle f(x)}$ is approximated using static quadrature rules on adaptively refined subintervals of the region of integration. Generally, adaptive algorithms are just as efficient and effective as traditional algorithms for "well behaved" integrands, but are also effective for "badly behaved" integrands for which traditional algorithms may fail.

## Extrapolation methods

The accuracy of a quadrature rule of the Newton–Cotes type is generally a function of the number of evaluation points. The result is usually more accurate as the number of evaluation points increases, or, equivalently, as the width of the step size between the points decreases. It is natural to ask what the result would be if the step size were allowed to approach zero. This can be answered by extrapolating the result from two or more nonzero step sizes, using series acceleration methods such as Richardson extrapolation. The extrapolation function may be a polynomial or rational function. Extrapolation methods are described in more detail by Stoer and Bulirsch (Section 3.4) and are implemented in many of the routines in the QUADPACK library.

## Conservative (a priori) error estimation

Let ${\displaystyle f}$ have a bounded first derivative over ${\displaystyle [a,b],}$ i.e. ${\displaystyle f\in C^{1}([a,b]).}$ The mean value theorem for ${\displaystyle f,}$ where ${\displaystyle x\in [a,b),}$ gives ${\displaystyle (x-a)f'(\xi _{x})=f(x)-f(a),}$ for some ${\displaystyle \xi _{x}\in (a,x]}$ depending on ${\displaystyle x}$.

If we integrate in ${\displaystyle x}$ from ${\displaystyle a}$ to ${\displaystyle b}$ on both sides and take the absolute values, we obtain ${\displaystyle \left|\int _{a}^{b}f(x)\,dx-(b-a)f(a)\right|=\left|\int _{a}^{b}(x-a)f'(\xi _{x})\,dx\right|.}$

We can further approximate the integral on the right-hand side by bringing the absolute value into the integrand, and replacing the term in ${\displaystyle f'}$ by an upper bound

$$ {\displaystyle \left|\int _{a}^{b}f(x)\,dx-(b-a)f(a)\right|\leq {(b-a)^{2} \over 2}\sup _{a\leq x\leq b}\left|f'(x)\right|,} $$

where the supremum was used to approximate.

Hence, if we approximate the integral ${\textstyle \int _{a}^{b}f(x)\,dx}$ by the quadrature rule ${\displaystyle (b-a)f(a)}$ our error is no greater than the right hand side of 1. We can convert this into an error analysis for the Riemann sum, giving an upper bound of ${\displaystyle {\frac {n^{-1}}{2}}\sup _{0\leq x\leq 1}\left|f'(x)\right|}$ for the error term of that particular approximation. (Note that this is precisely the error we calculated for the example ${\displaystyle f(x)=x}$.) Using more derivatives, and by tweaking the quadrature, we can do a similar error analysis using a Taylor series (using a partial sum with remainder term) for f. This error analysis gives a strict upper bound on the error, if the derivatives of f are available.

This integration method can be combined with interval arithmetic to produce computer proofs and verified calculations.

## Integrals over infinite intervals

Several methods exist for approximate integration over unbounded intervals. The standard technique involves specially derived quadrature rules, such as Gauss-Hermite quadrature for integrals on the whole real line and Gauss-Laguerre quadrature for integrals on the positive reals. Monte Carlo methods can also be used, or a change of variables to a finite interval; e.g., for the whole line one could use ${\displaystyle \int _{-\infty }^{\infty }f(x)\,dx=\int _{-1}^{+1}f\left({\frac {t}{1-t^{2}}}\right){\frac {1+t^{2}}{\left(1-t^{2}\right)^{2}}}\,dt,}$ and for semi-infinite intervals one could use ${\displaystyle {\begin{aligned}\int _{a}^{\infty }f(x)\,dx&=\int _{0}^{1}f\left(a+{\frac {t}{1-t}}\right){\frac {dt}{(1-t)^{2}}},\\\int _{-\infty }^{a}f(x)\,dx&=\int _{0}^{1}f\left(a-{\frac {1-t}{t}}\right){\frac {dt}{t^{2}}},\end{aligned}}}$ as possible transformations.

## Multidimensional integrals

The quadrature rules discussed so far are all designed to compute one-dimensional integrals. To compute integrals in multiple dimensions, one approach is to phrase the multiple integral as repeated one-dimensional integrals by applying Fubini's theorem (the tensor product rule). This approach requires the function evaluations to grow exponentially as the number of dimensions increases. Three methods are known to overcome this so-called curse of dimensionality.

A great many additional techniques for forming multidimensional cubature integration rules for a variety of weighting functions are given in the monograph by Stroud. Integration on the sphere has been reviewed by Hesse et al. (2015).

## Monte Carlo

Monte Carlo methods and quasi-Monte Carlo methods are easy to apply to multi-dimensional integrals. They may yield greater accuracy for the same number of function evaluations than repeated integrations using one-dimensional methods.

A large class of useful Monte Carlo methods are the so-called Markov chain Monte Carlo algorithms, which include the Metropolis–Hastings algorithm and Gibbs sampling.

## Sparse grids

Sparse grids were originally developed by Smolyak for the quadrature of high-dimensional functions. The method is always based on a one-dimensional quadrature rule, but performs a more sophisticated combination of univariate results. However, whereas the tensor product rule guarantees that the weights of all of the cubature points will be positive if the weights of the quadrature points were positive, Smolyak's rule does not guarantee that the weights will all be positive.

## Bayesian quadrature

Bayesian quadrature is a statistical approach to the numerical problem of computing integrals and falls under the field of probabilistic numerics. It can provide a full handling of the uncertainty over the solution of the integral expressed as a Gaussian process posterior variance.

## Connection with differential equations

The problem of evaluating the definite integral

$$ {\displaystyle F(x)=\int _{a}^{x}f(u)\,du} $$

can be reduced to an initial value problem for an ordinary differential equation by applying the first part of the fundamental theorem of calculus. By differentiating both sides of the above with respect to the argument x, it is seen that the function F satisfies

$$ {\displaystyle {\frac {dF(x)}{dx}}=f(x),\quad F(a)=0.} $$

Numerical methods for ordinary differential equations, such as Runge–Kutta methods, can be applied to the restated problem and thus be used to evaluate the integral. For instance, the standard fourth-order Runge–Kutta method applied to the differential equation yields Simpson's rule from above.

The differential equation ${\displaystyle F'(x)=f(x)}$ has a special form: the right-hand side contains only the independent variable (here ${\displaystyle x}$ ) and not the dependent variable (here ${\displaystyle F}$ ). This simplifies the theory and algorithms considerably. The problem of evaluating integrals is thus best studied in its own right.

Conversely, the term "quadrature" may also be used for the solution of differential equations: " solving by quadrature " or " reduction to quadrature " means expressing its solution in terms of integrals.


------------------------------------------------------------------

# test to pretty print simple wiki pages
# Issues: 
# latex printing 
# + FURTHER READINGS for Newton's method simple wiki page

In [8]:
# pretty_print_sections(simple_wiki_data[0]['sections'][0])
simple_wiki_data_test = simple_wiki_data[0]#['sections'][0]
selected_keys = ['url', 'title', 'sections']
filtered_data = {k: simple_wiki_data_test[k] for k in selected_keys}
filtered_data


{'url': 'https://simple.wikipedia.org/wiki/Derivative_(mathematics)',
 'title': 'Derivative (mathematics)',
 'sections': [{'heading': 'Introduction',
   'paragraphs': ['In mathematics (particularly in differential calculus ), the derivative is a way to show how steep a function is at a given point. Derivatives are similar to the slope of a line, but can be used for other curves as well. They are sometimes called the " instantaneous rate of change " of a function.',
    'More specifically, the derivative is how much a function is changing at one given point. For functions that act on the real numbers, it is the slope of the tangent line at a point on a graph. The derivative is often written as ${\\displaystyle {\\tfrac {dy}{dx}}}$ ("dy over dx" or "dy upon dx", meaning the difference in y divided by the difference in x). The d is not a variable, and therefore cannot be cancelled out. Another common notation is ${\\displaystyle f\'(x)}$ —the derivative of function ${\\displaystyle f}$ at

In [18]:
pretty_print_sections(filtered_data)

# Derivative (mathematics)

## Introduction

In mathematics (particularly in differential calculus ), the derivative is a way to show how steep a function is at a given point. Derivatives are similar to the slope of a line, but can be used for other curves as well. They are sometimes called the " instantaneous rate of change " of a function.

More specifically, the derivative is how much a function is changing at one given point. For functions that act on the real numbers, it is the slope of the tangent line at a point on a graph. The derivative is often written as ${\displaystyle {\tfrac {dy}{dx}}}$ ("dy over dx" or "dy upon dx", meaning the difference in y divided by the difference in x). The d is not a variable, and therefore cannot be cancelled out. Another common notation is ${\displaystyle f'(x)}$ —the derivative of function ${\displaystyle f}$ at point ${\displaystyle x}$, usually read as " ${\displaystyle f}$ prime of ${\displaystyle x}$ ".

## Definition of a derivative

The derivative of y with respect to x is defined as the change in y over the change in x, as the distance between ${\displaystyle x_{0}}$ and ${\displaystyle x_{1}}$ becomes infinitely small ( infinitesimal ). In mathematical terms,

${\displaystyle f'(a)=\lim _{h\to 0}{\frac {f(a+h)-f(a)}{h}}}$

That is, as the distance between the two x points (h) becomes closer to zero, the slope of the line between them comes closer to resembling a tangent line.

## Derivatives of functions

## Linear functions

Derivatives of linear functions (functions of the form ${\displaystyle mx+c}$ with no quadratic or higher terms) are constant. That is, the derivative in one spot on the graph will remain the same on another.

When the dependent variable ${\displaystyle y}$ directly takes ${\displaystyle x}$ 's value ( ${\displaystyle y=x}$ ), the slope of the line is 1 in all places, so ${\displaystyle {\tfrac {d}{dx}}(x)=1}$ regardless of where the position is.

When ${\displaystyle y}$ modifies ${\displaystyle x}$ 's number by adding or subtracting a constant value, the slope is still 1, because the change in ${\displaystyle x}$ and ${\displaystyle y}$ do not change if the graph is shifted up or down. That is, the slope is still 1 throughout the entire graph and its derivative is also 1.

## Power functions

Power functions (in the form of ${\displaystyle x^{a}}$ ) behave differently from linear functions, because their exponent and slope vary.

Power functions, in general, follow the rule that ${\displaystyle {\tfrac {d}{dx}}x^{a}=ax^{a-1}}$. That is, if we give a the number 6, then ${\displaystyle {\tfrac {d}{dx}}x^{6}=6x^{5}}$

Another example, which is less obvious, is the function ${\displaystyle f(x)={\tfrac {1}{x}}}$. This is essentially the same, because 1/x can be simplified to use exponents:

${\displaystyle f(x)={\frac {1}{x}}=x^{-1}}$

${\displaystyle f'(x)=-1(x^{-2})}$

${\displaystyle f'(x)=-{\frac {1}{x^{2}}}}$

In addition, roots can be changed to use fractional exponents, where their derivative can be found:

${\displaystyle f(x)={\sqrt[{3}]{x^{2}}}=x^{\frac {2}{3}}}$

${\displaystyle f'(x)={\frac {2}{3}}(x^{-{\frac {1}{3}}})}$

## Exponential functions

An exponential function is of the form ${\displaystyle ab^{f\left(x\right)}}$, where ${\displaystyle a}$ and ${\displaystyle b}$ are constants and ${\displaystyle f(x)}$ is a function of ${\displaystyle x}$. The difference between an exponential and a polynomial is that in a polynomial ${\displaystyle x}$ is raised to some power, whereas in an exponential ${\displaystyle x}$ is in the power.

## Example 1

${\displaystyle {\frac {d}{dx}}\left(ab^{f\left(x\right)}\right)=ab^{f(x)}\cdot f'\left(x\right)\cdot \ln(b)}$

## Example 2

Find ${\displaystyle {\frac {d}{dx}}\left(3\cdot 2^{3{x^{2}}}\right)}$.

${\displaystyle a=3}$

${\displaystyle b=2}$

${\displaystyle f\left(x\right)=3x^{2}}$

${\displaystyle f'\left(x\right)=6x}$

Therefore,

${\displaystyle {\frac {d}{dx}}\left(3\cdot 2^{3x^{2}}\right)=3\cdot 2^{3x^{2}}\cdot 6x\cdot \ln \left(2\right)=\ln \left(2\right)\cdot 18x\cdot 2^{3x^{2}}}$

## Logarithmic functions

The derivative of logarithms is the reciprocal:

${\displaystyle {\frac {d}{dx}}\ln(x)={\frac {1}{x}}}$.

Take, for example, ${\displaystyle {\frac {d}{dx}}\ln \left({\frac {5}{x}}\right)}$. This can be reduced to (by the properties of logarithms ):

${\displaystyle {\frac {d}{dx}}(\ln(5))-{\frac {d}{dx}}(\ln(x))}$

The logarithm of 5 is a constant, so its derivative is 0. The derivative of ${\displaystyle \ln(x)}$ is ${\displaystyle {\tfrac {1}{x}}}$. So,

${\displaystyle 0-{\frac {d}{dx}}\ln(x)=-{\frac {1}{x}}}$

For derivatives of logarithms not in base e, such as ${\displaystyle {\tfrac {d}{dx}}(\log _{10}(x))}$, this can be reduced to:

${\displaystyle {\frac {d}{dx}}\log _{10}(x)={\frac {d}{dx}}{\frac {\ln {x}}{\ln {10}}}={\frac {1}{\ln {10}}}{\frac {d}{dx}}\ln {x}={\frac {1}{x\ln(10)}}}$

## Trigonometric functions

The cosine function is the derivative of the sine function, while the derivative of cosine is negative sine (provided that x is measured in radians ):

## Properties of derivatives

Derivatives can be broken up into smaller parts where they are manageable (as they have only one of the above function characteristics). For example, ${\displaystyle {\tfrac {d}{dx}}(3x^{6}+x^{2}-6)}$ can be broken up as:

${\displaystyle {\frac {d}{dx}}(3x^{6})+{\frac {d}{dx}}(x^{2})-{\frac {d}{dx}}(6)}$

${\displaystyle =6\cdot 3x^{5}+2x-0}$

${\displaystyle =18x^{5}+2x\,}$

## Uses of derivatives

A function's derivative can be used to search for the maxima and minima of the function, by searching for places where its slope is zero.

Derivatives are used in Newton's method, which helps one find the zeros (roots) of a function. One can also use derivatives to determine the concavity of a function, and whether the function is increasing or decreasing.


------------------------------------------------------------------

In [14]:
test_page_title = ["Nim", "Newton's method", "Derivative (mathematics)"]
wiki_index = {page['title']: page for page in simple_wiki_data}

for title in test_page_title:
    selected_page = wiki_index.get(title)
    print(selected_page)


{'url': 'https://simple.wikipedia.org/wiki/Nim', 'title': 'Nim', 'sections': [{'heading': 'Introduction', 'paragraphs': ['Nim is a simple game used for examples in combinatorial game theory. The rules of nim are simple:', '1) The game begins with some piles of counters.\n2) Players alternate turns.\n3) On a turn, a player takes counters from a pile. At least one counter must be taken, but up to 3. All counters must be in the same pile.\n4) If a player cannot take a counter, that player loses.', 'There is a simple mathematical strategy to play the game perfectly. If both players play perfectly, the winner is determined by the initial setup.']}], 'categories': ['Game theory', 'Solved games'], 'category_urls': ['https://simple.wikipedia.org/wiki/Category:Game_theory', 'https://simple.wikipedia.org/wiki/Category:Solved_games'], 'last_scraped': '2026-01-28T10:40:00.372794+00:00', 'first_scraped': '2026-01-28T10:40:00.372794+00:00'}
{'url': 'https://simple.wikipedia.org/wiki/Newton%27s_metho

In [19]:
for title in test_page_title:
    selected_page = wiki_index.get(title)
    pretty_print_sections(selected_page)

# Nim

## Introduction

Nim is a simple game used for examples in combinatorial game theory. The rules of nim are simple:

1) The game begins with some piles of counters.
2) Players alternate turns.
3) On a turn, a player takes counters from a pile. At least one counter must be taken, but up to 3. All counters must be in the same pile.
4) If a player cannot take a counter, that player loses.

There is a simple mathematical strategy to play the game perfectly. If both players play perfectly, the winner is determined by the initial setup.


------------------------------------------------------------------

# Newton's method

## Introduction

Newton's method provides a way for finding the real zeros of a function. This algorithm is sometimes called the Newton–Raphson method, named after Sir Isaac Newton and Joseph Raphson.

The method uses the derivative of the function in order to find its roots. An initial "guess value" for the location of the zero must be made. From this value, a new guess is calculated by this formula:

${\displaystyle x_{n+1}=x_{n}-{\frac {f(x_{n})}{f'(x_{n})}}}$

Here x n is the initial guess and x n+1 is the next guess. The function f (whose zero is being solved for) has the derivative f'.

By repeatedly applying this formula to the generated guesses (that is by setting the value of x n to the formula's output and recomputing), the value of the guesses will approach a zero of the function.

Newton's method can be explained graphically by looking at intersections of tangent lines with the x-axis. First, a line tangent to the f at x n is calculated. Next, the intersection between this tangent line and the x-axis is found. Finally, the x-position of this intersection is recorded as the next guess, x n+1.

## Problems with Newton's Method

Newton's method can find a solution quickly if the guess value begins sufficiently near the desired root. However, when the initial guess value is not close, and depending on the function, Newton's method may find the answer slowly or not at all.


------------------------------------------------------------------

# Derivative (mathematics)

## Introduction

In mathematics (particularly in differential calculus ), the derivative is a way to show how steep a function is at a given point. Derivatives are similar to the slope of a line, but can be used for other curves as well. They are sometimes called the " instantaneous rate of change " of a function.

More specifically, the derivative is how much a function is changing at one given point. For functions that act on the real numbers, it is the slope of the tangent line at a point on a graph. The derivative is often written as ${\displaystyle {\tfrac {dy}{dx}}}$ ("dy over dx" or "dy upon dx", meaning the difference in y divided by the difference in x). The d is not a variable, and therefore cannot be cancelled out. Another common notation is ${\displaystyle f'(x)}$ —the derivative of function ${\displaystyle f}$ at point ${\displaystyle x}$, usually read as " ${\displaystyle f}$ prime of ${\displaystyle x}$ ".

## Definition of a derivative

The derivative of y with respect to x is defined as the change in y over the change in x, as the distance between ${\displaystyle x_{0}}$ and ${\displaystyle x_{1}}$ becomes infinitely small ( infinitesimal ). In mathematical terms,

${\displaystyle f'(a)=\lim _{h\to 0}{\frac {f(a+h)-f(a)}{h}}}$

That is, as the distance between the two x points (h) becomes closer to zero, the slope of the line between them comes closer to resembling a tangent line.

## Derivatives of functions

## Linear functions

Derivatives of linear functions (functions of the form ${\displaystyle mx+c}$ with no quadratic or higher terms) are constant. That is, the derivative in one spot on the graph will remain the same on another.

When the dependent variable ${\displaystyle y}$ directly takes ${\displaystyle x}$ 's value ( ${\displaystyle y=x}$ ), the slope of the line is 1 in all places, so ${\displaystyle {\tfrac {d}{dx}}(x)=1}$ regardless of where the position is.

When ${\displaystyle y}$ modifies ${\displaystyle x}$ 's number by adding or subtracting a constant value, the slope is still 1, because the change in ${\displaystyle x}$ and ${\displaystyle y}$ do not change if the graph is shifted up or down. That is, the slope is still 1 throughout the entire graph and its derivative is also 1.

## Power functions

Power functions (in the form of ${\displaystyle x^{a}}$ ) behave differently from linear functions, because their exponent and slope vary.

Power functions, in general, follow the rule that ${\displaystyle {\tfrac {d}{dx}}x^{a}=ax^{a-1}}$. That is, if we give a the number 6, then ${\displaystyle {\tfrac {d}{dx}}x^{6}=6x^{5}}$

Another example, which is less obvious, is the function ${\displaystyle f(x)={\tfrac {1}{x}}}$. This is essentially the same, because 1/x can be simplified to use exponents:

${\displaystyle f(x)={\frac {1}{x}}=x^{-1}}$

${\displaystyle f'(x)=-1(x^{-2})}$

${\displaystyle f'(x)=-{\frac {1}{x^{2}}}}$

In addition, roots can be changed to use fractional exponents, where their derivative can be found:

${\displaystyle f(x)={\sqrt[{3}]{x^{2}}}=x^{\frac {2}{3}}}$

${\displaystyle f'(x)={\frac {2}{3}}(x^{-{\frac {1}{3}}})}$

## Exponential functions

An exponential function is of the form ${\displaystyle ab^{f\left(x\right)}}$, where ${\displaystyle a}$ and ${\displaystyle b}$ are constants and ${\displaystyle f(x)}$ is a function of ${\displaystyle x}$. The difference between an exponential and a polynomial is that in a polynomial ${\displaystyle x}$ is raised to some power, whereas in an exponential ${\displaystyle x}$ is in the power.

## Example 1

${\displaystyle {\frac {d}{dx}}\left(ab^{f\left(x\right)}\right)=ab^{f(x)}\cdot f'\left(x\right)\cdot \ln(b)}$

## Example 2

Find ${\displaystyle {\frac {d}{dx}}\left(3\cdot 2^{3{x^{2}}}\right)}$.

${\displaystyle a=3}$

${\displaystyle b=2}$

${\displaystyle f\left(x\right)=3x^{2}}$

${\displaystyle f'\left(x\right)=6x}$

Therefore,

${\displaystyle {\frac {d}{dx}}\left(3\cdot 2^{3x^{2}}\right)=3\cdot 2^{3x^{2}}\cdot 6x\cdot \ln \left(2\right)=\ln \left(2\right)\cdot 18x\cdot 2^{3x^{2}}}$

## Logarithmic functions

The derivative of logarithms is the reciprocal:

${\displaystyle {\frac {d}{dx}}\ln(x)={\frac {1}{x}}}$.

Take, for example, ${\displaystyle {\frac {d}{dx}}\ln \left({\frac {5}{x}}\right)}$. This can be reduced to (by the properties of logarithms ):

${\displaystyle {\frac {d}{dx}}(\ln(5))-{\frac {d}{dx}}(\ln(x))}$

The logarithm of 5 is a constant, so its derivative is 0. The derivative of ${\displaystyle \ln(x)}$ is ${\displaystyle {\tfrac {1}{x}}}$. So,

${\displaystyle 0-{\frac {d}{dx}}\ln(x)=-{\frac {1}{x}}}$

For derivatives of logarithms not in base e, such as ${\displaystyle {\tfrac {d}{dx}}(\log _{10}(x))}$, this can be reduced to:

${\displaystyle {\frac {d}{dx}}\log _{10}(x)={\frac {d}{dx}}{\frac {\ln {x}}{\ln {10}}}={\frac {1}{\ln {10}}}{\frac {d}{dx}}\ln {x}={\frac {1}{x\ln(10)}}}$

## Trigonometric functions

The cosine function is the derivative of the sine function, while the derivative of cosine is negative sine (provided that x is measured in radians ):

## Properties of derivatives

Derivatives can be broken up into smaller parts where they are manageable (as they have only one of the above function characteristics). For example, ${\displaystyle {\tfrac {d}{dx}}(3x^{6}+x^{2}-6)}$ can be broken up as:

${\displaystyle {\frac {d}{dx}}(3x^{6})+{\frac {d}{dx}}(x^{2})-{\frac {d}{dx}}(6)}$

${\displaystyle =6\cdot 3x^{5}+2x-0}$

${\displaystyle =18x^{5}+2x\,}$

## Uses of derivatives

A function's derivative can be used to search for the maxima and minima of the function, by searching for places where its slope is zero.

Derivatives are used in Newton's method, which helps one find the zeros (roots) of a function. One can also use derivatives to determine the concavity of a function, and whether the function is increasing or decreasing.


------------------------------------------------------------------